<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Dailychallenge_W8D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Notebook - MCP + Airbnb (Colab)

Reference notebook: local notes MCP + Airbnb MCP + optional real LLM.

## Install
Run once. npm only needed for the real Airbnb server.

In [ ]:
!pip install -q mcp nest_asyncio requests
!pip install azure-ai-inference

# Optional: real Airbnb server
!npm install -g @openbnb/mcp-server-airbnb

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴npm warn deprecated whatwg-encoding@3.1.1: Use @exodus/bytes instead for a more spec-conformant and faster implementation
⠴⠦⠧⠇⠏npm warn deprecated node-domexception@1.0.0: Use your platform's native DOMException instead
⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
changed 124 packages in 5s
⠋
⠋51 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [ ]:
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    if self is sys.stderr:
        return 2
    return 1

OutStream.fileno = _patched_fileno
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [ ]:
import os
from pathlib import Path

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_AIRBNB = False  # True si vous avez npm et le serveur réel
USE_REAL_LLM = False     # True si vous avez GITHUB_TOKEN

In [ ]:
import os
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # pas de secret, on reste en stub

print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))

GITHUB_TOKEN visible to Python: True


## Local notes MCP server

In [ ]:
LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP
import sys
import logging

# Configuration des logs vers stderr pour ne pas polluer stdout (utilisé par MCP)
logging.basicConfig(level=logging.DEBUG, stream=sys.stderr)
logger = logging.getLogger(__name__)

mcp = FastMCP("local-notes")
notes = []

@mcp.tool()
def add_note(text: str) -> str:
    """Ajouter une note à la liste en mémoire."""
    notes.append(text)
    return f"Note sauvegardée: {text}"

@mcp.tool()
def list_notes() -> str:
    """Lister toutes les notes sauvegardées."""
    if not notes:
        return "Aucune note enregistrée."
    return "\\n".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    try:
        logger.info("Démarrage du serveur FastMCP...")
        mcp.run(transport='stdio')
    except Exception as e:
        logger.error(f"Erreur fatale du serveur: {e}")
        sys.exit(1)
'''.strip(),
    encoding="utf-8",
)
print(f"Fichier {LOCAL_SERVER} mis à jour avec logs de débogage.")

Fichier local_notes_server.py mis à jour avec logs de débogage.


## Stub Airbnb MCP server (fallback)

In [ ]:
STUB_AIRBNB = Path("stub_airbnb_server.py")
STUB_AIRBNB.write_text(
"""from mcp.server.fastmcp import FastMCP
import json

mcp = FastMCP("stub-airbnb")

FAKE_LISTINGS = {
    "Paris": [
        {"id": "p1", "name": "Studio Montmartre", "price": 120, "rating": 4.8},
        {"id": "p2", "name": "Appartement Louvre", "price": 180, "rating": 4.9},
    ],
    "New York": [
        {"id": "ny1", "name": "Loft Manhattan", "price": 250, "rating": 4.7},
        {"id": "ny2", "name": "Chambre Brooklyn", "price": 90, "rating": 4.5},
    ],
}

@mcp.tool()
def search_listings(location: str, check_in: str = "", check_out: str = "", guests: int = 2) -> str:
    '''Search for Airbnb listings in a city.'''
    listings = FAKE_LISTINGS.get(location, [])
    if not listings:
        return f"Aucune annonce trouvée pour {location}."
    return json.dumps(listings, indent=2)

@mcp.tool()
def get_listing_details(listing_id: str) -> str:
    '''Get details of a specific listing.'''
    for city, listings in FAKE_LISTINGS.items():
        for l in listings:
            if l["id"] == listing_id:
                return json.dumps({
                    "id": l["id"],
                    "name": l["name"],
                    "price": l["price"],
                    "rating": l["rating"],
                    "description": "Charmant logement en centre-ville.",
                    "amenities": ["WiFi", "Cuisine", "Chauffage"],
                }, indent=2)
    return f"Listing {listing_id} non trouvé."

if __name__ == "__main__":
    mcp.run()
""".strip(),
    encoding="utf-8"
)
print("wrote", STUB_AIRBNB)

wrote stub_airbnb_server.py


## Client helpers (convert tools, stub planner, optional real LLM)

In [ ]:
import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def convert_tool(tool, prefix: str):
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    if not use_real:
        # Stub planner simple
        lower = prompt.lower()
        calls = []
        if "note" in lower or "save" in lower:
            calls.append({"name": "notes__add_note", "args": {"text": prompt}})
        elif "list" in lower and "note" in lower:
            calls.append({"name": "notes__list_notes", "args": {}})
        elif any(city in lower for city in ["paris", "new york", "london"]):
            city = next((c for c in ["paris", "new york", "london"] if c in lower), "paris")
            calls.append({"name": "airbnb__search_listings", "args": {"location": city.title(), "guests": 2}})
        return calls

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token)
    )
    resp = client.complete(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that uses tools."},
            {"role": "user", "content": prompt}
        ],
        tools=functions,
        tool_choice="auto",
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [ ]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    import os
    import json

    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )

    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)

## Orchestrate (connect both servers and execute tool_calls)

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from contextlib import AsyncExitStack
import sys
import asyncio
from pathlib import Path

async def orchestrate(prompt: str):
    # Utilisation de sys.executable pour garantir le même environnement Python
    local_params = StdioServerParameters(
        command=sys.executable,
        args=[str(LOCAL_SERVER)],
        env=BASE_ENV,
    )

    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=BASE_ENV,
        )
    else:
        stub_script = Path("stub_airbnb_server.py")
        airbnb_params = StdioServerParameters(
            command=sys.executable,
            args=[str(stub_script)],
            env=BASE_ENV,
        )

    async with AsyncExitStack() as stack:
        try:
            print("Démarrage du serveur local...")
            l_stdio = await stack.enter_async_context(stdio_client(local_params, errlog=sys.stderr))
            local_sess = await stack.enter_async_context(ClientSession(l_stdio[0], l_stdio[1]))
            await asyncio.sleep(1)
            await local_sess.initialize()
            local_tools = await local_sess.list_tools()
        except Exception as e:
            print(f"Erreur serveur Local: {e}")
            raise

        try:
            print("Démarrage du serveur Airbnb...")
            a_stdio = await stack.enter_async_context(stdio_client(airbnb_params, errlog=sys.stderr))
            airbnb_sess = await stack.enter_async_context(ClientSession(a_stdio[0], a_stdio[1]))
            await asyncio.sleep(1)
            await airbnb_sess.initialize()
            airbnb_tools = await airbnb_sess.list_tools()
        except Exception as e:
            print(f"Erreur serveur Airbnb: {e}")
            raise

        functions = (
            [convert_tool(t, "notes") for t in local_tools.tools]
            + [convert_tool(t, "airbnb") for t in airbnb_tools.tools]
        )

        tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
        print("tool_calls:", tool_calls)

        tool_results = []
        for call in tool_calls:
            name = call["name"]
            args = call["args"]
            prefix, tool_name = name.split("__", 1)

            if prefix == "notes":
                res = await local_sess.call_tool(tool_name, args)
                tool_results.append({
                    "name": name, "args": args,
                    "content": [c.text for c in res.content if hasattr(c, "text")]
                })
            elif prefix == "airbnb":
                res = await airbnb_sess.call_tool(tool_name, args)
                tool_results.append({
                    "name": name, "args": args,
                    "content": [c.text for c in res.content if hasattr(c, "text")]
                })

        return tool_calls, tool_results

## Demo
Adjust the prompt as you like. Switch `USE_REAL_AIRBNB/USE_REAL_LLM` to true when ready.

In [ ]:
prompt = "Je cherche un logement à Paris pour 2 personnes, et je veux sauvegarder une note avec les résultats."

try:
    # Appel de l'orchestrateur
    tool_calls, tool_results = await orchestrate(prompt)

    print("\n--- Tool calls ---")
    print(json.dumps(tool_calls, indent=2))
    print("\n--- Tool results ---")
    print(json.dumps(tool_results, indent=2))

    # Si le LLM réel est activé, on génère une réponse finale
    if USE_REAL_LLM:
        answer = answer_with_llm(prompt, tool_calls, tool_results, use_real=True)
        print("\n--- Réponse finale ---")
        print(answer)
    else:
        print("\n--- Mode stub : pas de réponse LLM générée ---")
except Exception as e:
    print(f"\nL'erreur persiste lors de l'orchestration : {e}")
    print("Vérifiez que la cellule IxooA6XWWPNq a bien été exécutée pour recréer le fichier serveur.")

Démarrage du serveur local...
Démarrage du serveur Airbnb...
tool_calls: [{'name': 'notes__add_note', 'args': {'text': 'Je cherche un logement à Paris pour 2 personnes, et je veux sauvegarder une note avec les résultats.'}}]

--- Tool calls ---
[
  {
    "name": "notes__add_note",
    "args": {
      "text": "Je cherche un logement \u00e0 Paris pour 2 personnes, et je veux sauvegarder une note avec les r\u00e9sultats."
    }
  }
]

--- Tool results ---
[
  {
    "name": "notes__add_note",
    "args": {
      "text": "Je cherche un logement \u00e0 Paris pour 2 personnes, et je veux sauvegarder une note avec les r\u00e9sultats."
    },
    "content": [
      "Note sauvegard\u00e9e: Je cherche un logement \u00e0 Paris pour 2 personnes, et je veux sauvegarder une note avec les r\u00e9sultats."
    ]
  }
]

--- Mode stub : pas de réponse LLM générée ---
